# Assign weather files to sampled cities

Each sampled city is assigned one EPW typical meteorological year file and one
DDY design-day file. Both are named with the city's `city_key` and are used
by `BEM4AI_7_Simulation.ipynb`.

## Obtain the weather archives

Climate.OneBuilding weather archives are not redistributed with BEM4AI and must
be downloaded manually. The station catalogue is available from
[Climate.OneBuilding](https://climate.onebuilding.org/)
([direct download](https://climate.onebuilding.org/sources/Region6_Europe_TMYx_EPW_Processing_locations.xlsx)).

**Inputs**
- `data/climate_onebuilding/Region6_Europe_TMYx_EPW_Processing_locations.xlsx`
- `data/interim/city_assets_df.pkl`
- `data/interim/buildings_gdf.pkl`

**Steps**
1. Run the matching and download-list cells.
2. Download each distinct archive listed in `data/interim/weather_download_links.csv` to
   `output/weather/downloads/`, retaining its original filename.
3. Run the extraction and registry cells.

**Outputs**
- `data/interim/weather_download_links.csv`
- `output/weather/epw/<city_key>.epw` and `output/weather/ddy/<city_key>.ddy`
- `data/interim/city_weather_assets.pkl`

Climate.OneBuilding periodically refreshes station archives. The filenames and
directory layout are reproducible, but historical archive contents may differ.


## Prepare weather assignments

### Load sampled cities and the station catalogue

**Inputs**
- `data/climate_onebuilding/Region6_Europe_TMYx_EPW_Processing_locations.xlsx`
- `data/interim/city_assets_df.pkl`
- `data/interim/buildings_gdf.pkl`

**Steps**
1. Load the station catalogue and the Stage 1 city-asset table.
2. Restrict the city table to cities represented by Stage 3 building samples.

**Outputs**: `climate_onebuilding_df` and `city_weather_assets`.


In [1]:
import sys
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from helpers.paths import PROJECT_ROOT, DATA_DIR, INTERIM_DIR
from helpers.utils import haversine_m

CLIMATE_DIR = DATA_DIR / "climate_onebuilding"
CITY_ASSETS_PKL = INTERIM_DIR / "city_assets_df.pkl"
BUILDINGS_PKL = INTERIM_DIR / "buildings_gdf.pkl"
CITY_WEATHER_ASSETS_PKL = INTERIM_DIR / "city_weather_assets.pkl"
LINKS_CSV = INTERIM_DIR / "weather_download_links.csv"

WEATHER_DIR = PROJECT_ROOT / "output" / "weather"
DOWNLOADS_DIR = WEATHER_DIR / "downloads"
EPW_DIR = WEATHER_DIR / "epw"
DDY_DIR = WEATHER_DIR / "ddy"
EXTRACT_SCRIPT = PROJECT_ROOT / "scripts" / "extract_weather.py"

# Station catalogue (manual download, see the note at the top)
XLSX_PATH = CLIMATE_DIR / "Region6_Europe_TMYx_EPW_Processing_locations.xlsx"
if not XLSX_PATH.exists():
    raise FileNotFoundError(
        f"Station catalogue not found: {XLSX_PATH}\n\n"
        "This file is not redistributed with BEM4AI. Download\n"
        "`Region6_Europe_TMYx_EPW_Processing_locations.xlsx` from\n"
        "https://climate.onebuilding.org/sources/Region6_Europe_TMYx_EPW_Processing_locations.xlsx\nand save it as the path above, then re-run this cell."
    )

xlsx = pd.ExcelFile(XLSX_PATH)
climate_onebuilding_df = pd.concat(
    [pd.read_excel(XLSX_PATH, sheet_name=s) for s in xlsx.sheet_names],
    ignore_index=True,
)

# Restrict the Stage 1 registry to cities with Stage 3 building samples
required_paths = [CITY_ASSETS_PKL, BUILDINGS_PKL]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required artifact(s): " + ", ".join(missing_paths) +
        ". Run BEM4AI_1_Cities.ipynb and BEM4AI_3_Buildings.ipynb first."
    )

city_assets_df = pd.read_pickle(CITY_ASSETS_PKL)
buildings_gdf = pd.read_pickle(BUILDINGS_PKL)
if not isinstance(city_assets_df, pd.DataFrame):
    raise TypeError(f"{CITY_ASSETS_PKL.name} must contain one pandas DataFrame.")
if not isinstance(buildings_gdf, pd.DataFrame):
    raise TypeError(f"{BUILDINGS_PKL.name} must contain one pandas DataFrame.")
if "city_key" not in city_assets_df.columns or "city_key" not in buildings_gdf.columns:
    raise KeyError("Both city_assets_df and buildings_gdf must contain 'city_key'.")

sampled_city_keys = sorted(buildings_gdf["city_key"].dropna().astype(str).unique())
if not sampled_city_keys:
    raise ValueError(f"{BUILDINGS_PKL.name} does not contain any sampled city keys.")

city_assets_df = city_assets_df.copy()
city_assets_df["city_key"] = city_assets_df["city_key"].astype(str)
missing_city_keys = sorted(set(sampled_city_keys) - set(city_assets_df["city_key"]))
if missing_city_keys:
    raise KeyError(
        f"{len(missing_city_keys)} sampled cities are absent from {CITY_ASSETS_PKL.name}: "
        + ", ".join(missing_city_keys[:10])
    )

city_weather_assets = (
    city_assets_df[city_assets_df["city_key"].isin(sampled_city_keys)]
    .drop_duplicates(subset=["city_key"])
    .sort_values("city_key")
    .reset_index(drop=True)
)

print(f"Station catalogue sheets: {xlsx.sheet_names}")
print(f"Stations listed:          {len(climate_onebuilding_df)}")
print(f"Sampled buildings:        {len(buildings_gdf)}")
print(f"Sampled cities:           {len(city_weather_assets)}")
city_weather_assets.head()


Station catalogue sheets: ['Reg6_Europe_TMYx_EPW_Processing']
Stations listed:          16286
Sampled buildings:        18
Sampled cities:           16


,city_ghsl,country,lat,lon,city_key,country_key,osm_pbf,builtc_tif,builth_tif,builtage_tif
0,Athens,Greece,37.993404,23.736704,Athens,Greece,Athens.osm.pbf,Athens_GHS_BUILT_C_2x2km_moll.tif,Athens_GHS_BUILT_H_2x2km_moll.tif,Athens_GHS_AGE_2x2km_moll.tif
1,Berlin,Germany,52.503040,13.400561,Berlin,Germany,Berlin.osm.pbf,Berlin_GHS_BUILT_C_2x2km_moll.tif,Berlin_GHS_BUILT_H_2x2km_moll.tif,Berlin_GHS_AGE_2x2km_moll.tif
2,Birmingham,United Kingdom,52.510819,-1.961703,Birmingham,United_Kingdom,Birmingham.osm.pbf,Birmingham_GHS_BUILT_C_2x2km_moll.tif,Birmingham_GHS_BUILT_H_2x2km_moll.tif,Birmingham_GHS_AGE_2x2km_moll.tif
3,Bucharest,Romania,44.435108,26.100913,Bucharest,Romania,Bucharest.osm.pbf,Bucharest_GHS_BUILT_C_2x2km_moll.tif,Bucharest_GHS_BUILT_H_2x2km_moll.tif,Bucharest_GHS_AGE_2x2km_moll.tif
4,Budapest,Hungary,47.487474,19.090479,Budapest,Hungary,Budapest.osm.pbf,Budapest_GHS_BUILT_C_2x2km_moll.tif,Budapest_GHS_BUILT_H_2x2km_moll.tif,Budapest_GHS_AGE_2x2km_moll.tif


### Match sampled cities to weather stations

Weather archives are not matched by city name. Station names can use local
languages, several stations can serve the same city, and some sampled cities
have no station. Coordinate matching therefore selects the closest available
station for every sampled city.

**Inputs**: `city_weather_assets` and `climate_onebuilding_df`.

**Steps**
1. Prepare station coordinates and archive URLs.
2. Consolidate duplicate station records to one preferred archive URL.
3. Select the closest station for each sampled city from geographic distance.
4. Add the station name, distance, and archive URL to `city_weather_assets`.

**Outputs**: `city_weather_assets` with `weather_station`,
`weather_station_distance_m`, and `weather_url`.


In [2]:
# Required columns
CITY_LAT_COL = "lat"
CITY_LON_COL = "lon"
STN_LAT_COL = "Latitude (N+/S-)"
STN_LON_COL = "Longitude (E+/W-)"
STN_NAME_COL = "City/Station"
STN_URL_COL = "URL"

# Verify the location fields required for matching
missing_city_cols = [
    column for column in (CITY_LAT_COL, CITY_LON_COL)
    if column not in city_weather_assets.columns
]
missing_stn_cols = [
    column for column in (STN_LAT_COL, STN_LON_COL, STN_NAME_COL, STN_URL_COL)
    if column not in climate_onebuilding_df.columns
]
if missing_city_cols:
    raise KeyError(
        f"city_weather_assets is missing required columns: {missing_city_cols}"
    )
if missing_stn_cols:
    raise KeyError(
        f"climate_onebuilding_df is missing required columns: {missing_stn_cols}"
    )

# Convert coordinates and archive URLs to values suitable for distance matching
cities_work = city_weather_assets.copy()
cities_work[CITY_LAT_COL] = pd.to_numeric(
    cities_work[CITY_LAT_COL], errors="coerce"
)
cities_work[CITY_LON_COL] = pd.to_numeric(
    cities_work[CITY_LON_COL], errors="coerce"
)

stations_work = climate_onebuilding_df[
    [STN_NAME_COL, STN_LAT_COL, STN_LON_COL, STN_URL_COL]
].copy()
stations_work[STN_LAT_COL] = pd.to_numeric(
    stations_work[STN_LAT_COL], errors="coerce"
)
stations_work[STN_LON_COL] = pd.to_numeric(
    stations_work[STN_LON_COL], errors="coerce"
)
stations_work[STN_URL_COL] = stations_work[STN_URL_COL].astype("string").str.strip()
stations_work = stations_work.dropna(
    subset=[STN_LAT_COL, STN_LON_COL, STN_URL_COL]
)
stations_work = stations_work.loc[stations_work[STN_URL_COL].ne("")].copy()
if stations_work.empty:
    raise ValueError(
        "The station catalogue contains no records with coordinates and archive URLs."
    )

# Some stations have several archive URLs. Prefer URLs without a year range.
def _choose_best_url(urls: pd.Series) -> str:
    return min(
        (str(url) for url in urls),
        key=lambda url: (
            bool(re.search(r"\.\d{4}-\d{4}\.", Path(urlparse(url).path).name)),
            len(Path(urlparse(url).path).name),
            Path(urlparse(url).path).name,
        ),
    )

# Retain one preferred archive URL for each station location
grouped = stations_work.groupby(
    [STN_NAME_COL, STN_LAT_COL, STN_LON_COL], dropna=False
)
stations_valid = (
    grouped[STN_URL_COL]
    .apply(_choose_best_url)
    .reset_index(name=STN_URL_COL)
    .reset_index(drop=True)
)

# Keep original row positions so station matches can be attached to the registry
cities_valid = (
    cities_work
    .dropna(subset=[CITY_LAT_COL, CITY_LON_COL])
    .reset_index(drop=False)
    .rename(columns={"index": "_orig_idx"})
)
if cities_valid.empty:
    raise ValueError("No sampled cities have valid latitude and longitude values.")

print(f"Sampled cities:             {len(city_weather_assets)}")
print(f"Cities with coordinates:    {len(cities_valid)}")
print(f"Available weather stations: {len(stations_valid)}")

city_lat_rad = np.radians(cities_valid[CITY_LAT_COL].to_numpy())
city_lon_rad = np.radians(cities_valid[CITY_LON_COL].to_numpy())
stn_lat_rad = np.radians(stations_valid[STN_LAT_COL].to_numpy())
stn_lon_rad = np.radians(stations_valid[STN_LON_COL].to_numpy())

n_cities = len(cities_valid)
n_stations = len(stations_valid)
max_distance_matrix_entries = 20_000_000
city_batch_size = max(
    1, min(2_000, max_distance_matrix_entries // max(1, n_stations))
)

nearest_idx = np.empty(n_cities, dtype=int)
nearest_dist_m = np.empty(n_cities, dtype=float)

# Process cities in batches to bound the distance-matrix memory requirement
for start in range(0, n_cities, city_batch_size):
    end = min(start + city_batch_size, n_cities)
    city_lat_batch = city_lat_rad[start:end, None]
    city_lon_batch = city_lon_rad[start:end, None]

    distances_m = haversine_m(
        city_lat_batch,
        city_lon_batch,
        stn_lat_rad[None, :],
        stn_lon_rad[None, :],
    )
    closest_station = np.argmin(distances_m, axis=1)
    nearest_idx[start:end] = closest_station
    nearest_dist_m[start:end] = distances_m[
        np.arange(end - start), closest_station
    ]

# Attach the selected station to each sampled city
matched_stations = stations_valid.iloc[nearest_idx].reset_index(drop=True)
matched_rows = cities_valid["_orig_idx"].to_numpy()
city_weather_assets = city_weather_assets.copy()
city_weather_assets.loc[
    matched_rows, "weather_station"
] = matched_stations[STN_NAME_COL].to_numpy()
city_weather_assets.loc[
    matched_rows, "weather_station_distance_m"
] = nearest_dist_m
city_weather_assets.loc[
    matched_rows, "weather_url"
] = matched_stations[STN_URL_COL].to_numpy()

city_weather_assets


Sampled cities:             16
Cities with coordinates:    16
Available weather stations: 4445


,city_ghsl,country,lat,lon,city_key,country_key,osm_pbf,builtc_tif,builth_tif,builtage_tif,weather_station,weather_station_distance_m,weather_url
0,Athens,Greece,37.993404,23.736704,Athens,Greece,Athens.osm.pbf,Athens_GHS_BUILT_C_2x2km_moll.tif,Athens_GHS_BUILT_H_2x2km_moll.tif,Athens_GHS_AGE_2x2km_moll.tif,Athinai-Hellinikon.Olympic.Complex,11539.653565,https://climate.onebuilding.org/WMO_Region_6_E...
1,Berlin,Germany,52.503040,13.400561,Berlin,Germany,Berlin.osm.pbf,Berlin_GHS_BUILT_C_2x2km_moll.tif,Berlin_GHS_BUILT_H_2x2km_moll.tif,Berlin_GHS_AGE_2x2km_moll.tif,Berlin-Tempelhof.AP,3953.476004,https://climate.onebuilding.org/WMO_Region_6_E...
2,Birmingham,United Kingdom,52.510819,-1.961703,Birmingham,United_Kingdom,Birmingham.osm.pbf,Birmingham_GHS_BUILT_C_2x2km_moll.tif,Birmingham_GHS_BUILT_H_2x2km_moll.tif,Birmingham_GHS_AGE_2x2km_moll.tif,Birmingham.AP,15790.651775,https://climate.onebuilding.org/WMO_Region_6_E...
3,Bucharest,Romania,44.435108,26.100913,Bucharest,Romania,Bucharest.osm.pbf,Bucharest_GHS_BUILT_C_2x2km_moll.tif,Bucharest_GHS_BUILT_H_2x2km_moll.tif,Bucharest_GHS_AGE_2x2km_moll.tif,Bucharest,2640.072504,https://climate.onebuilding.org/WMO_Region_6_E...
4,Budapest,Hungary,47.487474,19.090479,Budapest,Hungary,Budapest.osm.pbf,Budapest_GHS_BUILT_C_2x2km_moll.tif,Budapest_GHS_BUILT_H_2x2km_moll.tif,Budapest_GHS_AGE_2x2km_moll.tif,Budapest.Met.Center,5424.210340,https://climate.onebuilding.org/WMO_Region_6_E...
5,Essen,Germany,51.490183,6.968052,Essen,Germany,Essen.osm.pbf,Essen_GHS_BUILT_C_2x2km_moll.tif,Essen_GHS_BUILT_H_2x2km_moll.tif,Essen_GHS_AGE_2x2km_moll.tif,Essen.DWD,9560.867746,https://climate.onebuilding.org/WMO_Region_6_E...
6,Lisbon,Portugal,38.763393,-9.226947,Lisbon,Portugal,Lisbon.osm.pbf,Lisbon_GHS_BUILT_C_2x2km_moll.tif,Lisbon_GHS_BUILT_H_2x2km_moll.tif,Lisbon_GHS_AGE_2x2km_moll.tif,Lisboa.Portela.AP,8123.841449,https://climate.onebuilding.org/WMO_Region_6_E...
7,London,United Kingdom,51.504729,-0.133608,London,United_Kingdom,London.osm.pbf,London_GHS_BUILT_C_2x2km_moll.tif,London_GHS_BUILT_H_2x2km_moll.tif,London_GHS_AGE_2x2km_moll.tif,London.Wea.Ctr-St.James.Park,181.538807,https://climate.onebuilding.org/WMO_Region_6_E...
8,Madrid,Spain,40.402321,-3.739645,Madrid,Spain,Madrid.osm.pbf,Madrid_GHS_BUILT_C_2x2km_moll.tif,Madrid_GHS_BUILT_H_2x2km_moll.tif,Madrid_GHS_AGE_2x2km_moll.tif,Madrid-Cuatro.Vientos.AP,4937.045505,https://climate.onebuilding.org/WMO_Region_6_E...
9,Manchester,United Kingdom,53.498334,-2.265721,Manchester,United_Kingdom,Manchester.osm.pbf,Manchester_GHS_BUILT_C_2x2km_moll.tif,Manchester_GHS_BUILT_H_2x2km_moll.tif,Manchester_GHS_AGE_2x2km_moll.tif,Manchester-Barton,1997.181895,https://climate.onebuilding.org/WMO_Region_6_E...


## Create the download list

### Write weather archive links

**Inputs**: `city_weather_assets` with `city_key` and `weather_url`.

**Steps**
1. Create one weather archive link per sampled city.
2. Save the links and report which distinct archives are still missing.

**Outputs**: `data/interim/weather_download_links.csv` and
`output/weather/downloads/`.


In [3]:
# Write one archive URL for each sampled city
required = {"city_key", "weather_url"}
missing = required - set(city_weather_assets.columns)
if missing:
    raise KeyError(
        f"city_weather_assets is missing required columns: {sorted(missing)}"
    )

links_df = (
    city_weather_assets[["city_key", "weather_url"]]
    .rename(columns={"weather_url": "download_link"})
    .dropna(subset=["download_link"])
    .astype({"city_key": str, "download_link": str})
)
links_df["download_link"] = links_df["download_link"].str.strip()
links_df = links_df[links_df["download_link"] != ""]

without_link = len(city_weather_assets) - len(links_df)
if without_link:
    print(f"{without_link} city or cities have no matched weather archive.")

# Verify that the selected URLs point to ZIP archives
links_df["archive_name"] = links_df["download_link"].map(
    lambda url: Path(urlparse(url).path).name
)
bad_links = links_df[
    ~links_df["archive_name"].str.lower().str.endswith(".zip")
]
if not bad_links.empty:
    raise ValueError(
        "These matched station URLs do not point to ZIP archives:\n"
        + bad_links[["city_key", "download_link"]].to_string(index=False)
    )

links_df = links_df.drop_duplicates(subset="city_key").sort_values("city_key")
LINKS_CSV.parent.mkdir(parents=True, exist_ok=True)
links_df[["city_key", "download_link"]].to_csv(LINKS_CSV, index=False)

# Compare the required archives with the downloaded files
DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)
required_archives = set(links_df["archive_name"])
downloaded_archives = {path.name for path in DOWNLOADS_DIR.glob("*.zip")}
missing_archives = sorted(required_archives - downloaded_archives)

print(f"City links written:        {len(links_df)}")
print(f"Distinct archives needed:  {len(required_archives)}")
print(f"Archives already present:  {len(required_archives) - len(missing_archives)}")
print(f"Archives still missing:    {len(missing_archives)}")

if missing_archives:
    print(
        "\nDownload the missing archives listed in "
        f"{LINKS_CSV.relative_to(PROJECT_ROOT)} to "
        f"{DOWNLOADS_DIR.relative_to(PROJECT_ROOT)}/ before continuing."
    )
    print("\nFirst missing archives:")
    for archive_name in missing_archives[:5]:
        print(f"  - {archive_name}")
    if len(missing_archives) > 5:
        print(f"  ... and {len(missing_archives) - 5} more")
else:
    print("\nAll required archives are present.")


City links written:        16
Distinct archives needed:  16
Archives already present:  16
Archives still missing:    0

All required archives are present.


## Validate downloaded archives

### Check archive contents

Validate every archive listed in `weather_download_links.csv` before
extracting files. Each archive must be readable and contain one usable EPW file
and one usable DDY file.

**Outputs**: a validation report; no files are written.


In [4]:
def run_extract_weather(*args: str) -> int:
    "Run the weather extraction script and display its output."
    if not EXTRACT_SCRIPT.exists():
        raise FileNotFoundError(f"Extraction helper not found: {EXTRACT_SCRIPT}")
    command = [
        sys.executable,
        str(EXTRACT_SCRIPT),
        "--links",
        str(LINKS_CSV),
        "--downloads-dir",
        str(DOWNLOADS_DIR),
        "--epw-dir",
        str(EPW_DIR),
        "--ddy-dir",
        str(DDY_DIR),
        *args,
    ]
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    return completed.returncode

validation_status = run_extract_weather("--check-only")
if validation_status != 0:
    raise RuntimeError(
        "Weather archive validation failed. Download the missing archives and rerun this cell."
    )


City outputs expected: 16
ZIP archives expected: 16
ZIP archives valid:    16
Check complete; no EPW/DDY files were written.


### Extract EPW and DDY files

Extract one EPW/DDY pair per sampled city. Cities sharing a station receive
separate files named by their `city_key`. Existing complete outputs are retained
unless `FORCE_REEXTRACT` is set to `True`.

**Outputs**: `output/weather/epw/` and `output/weather/ddy/`.


In [5]:
# Set to True only to replace complete existing weather outputs
FORCE_REEXTRACT = False

expected_epw = {f"{city_key}.epw" for city_key in links_df["city_key"]}
expected_ddy = {f"{city_key}.ddy" for city_key in links_df["city_key"]}
have_epw = {path.name for path in EPW_DIR.glob("*.epw")} if EPW_DIR.is_dir() else set()
have_ddy = {path.name for path in DDY_DIR.glob("*.ddy")} if DDY_DIR.is_dir() else set()

already_complete = expected_epw <= have_epw and expected_ddy <= have_ddy
if already_complete and not FORCE_REEXTRACT:
    print(f"Weather files already exist for all {len(expected_epw)} sampled cities.")
else:
    missing_epw = len(expected_epw - have_epw)
    missing_ddy = len(expected_ddy - have_ddy)
    print(f"Extracting {missing_epw} missing EPW and {missing_ddy} missing DDY files.")
    extraction_status = run_extract_weather("--force")
    if extraction_status != 0:
        raise RuntimeError(
            "Weather extraction failed. Check the archive validation message above."
        )


Extracting 16 missing EPW and 16 missing DDY files.
City outputs expected: 16
ZIP archives expected: 16
ZIP archives valid:    16
EPW files written:     16 -> output/weather/epw
DDY files written:     16 -> output/weather/ddy


## Save the weather registry

### Record selected weather files

**Inputs**: `city_weather_assets` and the extracted EPW/DDY files.

**Steps**
1. Add the weather filenames and repository-relative paths.
2. Confirm that every selected file exists.
3. Save the registry for Stage 7.

**Outputs**: `data/interim/city_weather_assets.pkl`, joined to models by
`city_key` in `BEM4AI_7_Simulation.ipynb`.


In [6]:
# Add filenames and repository-relative paths for Stage 7
if "city_key" not in city_weather_assets.columns:
    raise KeyError("city_weather_assets is missing required column 'city_key'.")

city_weather_assets = city_weather_assets.copy()
city_weather_assets["city_key"] = city_weather_assets["city_key"].astype(str)
city_weather_assets["epw_file"] = city_weather_assets["city_key"] + ".epw"
city_weather_assets["ddy_file"] = city_weather_assets["city_key"] + ".ddy"
city_weather_assets["epw_relpath"] = [
    (EPW_DIR / filename).relative_to(PROJECT_ROOT).as_posix()
    for filename in city_weather_assets["epw_file"]
]
city_weather_assets["ddy_relpath"] = [
    (DDY_DIR / filename).relative_to(PROJECT_ROOT).as_posix()
    for filename in city_weather_assets["ddy_file"]
]

# Confirm all selected weather files exist before saving the registry
missing_files = [
    (row.city_key, kind)
    for row in city_weather_assets.itertuples()
    for kind, directory, filename in (
        ("epw", EPW_DIR, row.epw_file),
        ("ddy", DDY_DIR, row.ddy_file),
    )
    if not (directory / filename).is_file()
]
if missing_files:
    preview = ", ".join(f"{city_key}.{kind}" for city_key, kind in missing_files[:10])
    more = f" (and {len(missing_files) - 10} more)" if len(missing_files) > 10 else ""
    raise FileNotFoundError(
        f"{len(missing_files)} selected weather file(s) are missing: {preview}{more}"
    )

city_weather_assets.to_pickle(CITY_WEATHER_ASSETS_PKL)
print(f"Verified {len(city_weather_assets)} EPW/DDY pairs.")
print(f"Weather registry: {CITY_WEATHER_ASSETS_PKL.relative_to(PROJECT_ROOT)}")
city_weather_assets[
    [
        "city_key",
        "weather_station",
        "weather_station_distance_m",
        "epw_relpath",
        "ddy_relpath",
    ]
].head()


Verified 16 EPW/DDY pairs.
Weather registry: data/interim/city_weather_assets.pkl


,city_key,weather_station,weather_station_distance_m,epw_relpath,ddy_relpath
0,Athens,Athinai-Hellinikon.Olympic.Complex,11539.653565,output/weather/epw/Athens.epw,output/weather/ddy/Athens.ddy
1,Berlin,Berlin-Tempelhof.AP,3953.476004,output/weather/epw/Berlin.epw,output/weather/ddy/Berlin.ddy
2,Birmingham,Birmingham.AP,15790.651775,output/weather/epw/Birmingham.epw,output/weather/ddy/Birmingham.ddy
3,Bucharest,Bucharest,2640.072504,output/weather/epw/Bucharest.epw,output/weather/ddy/Bucharest.ddy
4,Budapest,Budapest.Met.Center,5424.210340,output/weather/epw/Budapest.epw,output/weather/ddy/Budapest.ddy
